In [1]:
import pandas as pd
import time
import csv
import re
comp=pd.read_csv('glycaninsilicowithkdn.csv', header =None)
sortcomp = comp.sort_values([1]).reset_index(drop=True)

In [2]:
#testing block

print("Beofre sorting")
for i in range(3):
    print(comp[1][i])
print("After sorting")
for i in range(22):
    print(str(sortcomp[1][i]) + " mass <-, composition ->" + str(sortcomp[0][i]))


Beofre sorting
1149.601
1484.771
1819.941
After sorting
1149.601 mass <-, composition ->(0, 3, 2, 0, 0, 0)
1323.69 mass <-, composition ->(1, 3, 2, 0, 0, 0)
1353.701 mass <-, composition ->(0, 4, 2, 0, 0, 0)
1394.727 mass <-, composition ->(0, 3, 3, 0, 0, 0)
1484.771 mass <-, composition ->(0, 3, 2, 0, 0, 1)
1497.78 mass <-, composition ->(2, 3, 2, 0, 0, 0)
1510.775 mass <-, composition ->(0, 3, 2, 1, 0, 0)
1527.79 mass <-, composition ->(1, 4, 2, 0, 0, 0)
1540.785 mass <-, composition ->(0, 3, 2, 0, 1, 0)
1557.801 mass <-, composition ->(0, 5, 2, 0, 0, 0)
1568.817 mass <-, composition ->(1, 3, 3, 0, 0, 0)
1598.827 mass <-, composition ->(0, 4, 3, 0, 0, 0)
1639.854 mass <-, composition ->(0, 3, 4, 0, 0, 0)
1658.86 mass <-, composition ->(1, 3, 2, 0, 0, 1)
1671.869 mass <-, composition ->(3, 3, 2, 0, 0, 0)
1684.864 mass <-, composition ->(1, 3, 2, 1, 0, 0)
1688.871 mass <-, composition ->(0, 4, 2, 0, 0, 1)
1701.879 mass <-, composition ->(2, 4, 2, 0, 0, 0)
1714.875 mass <-, composition 

In [3]:
#modified from programmiz
def binarycompositionsearch(array, x, low, high, boundary):
    # Repeat until the pointers low and high meet each other
    while low <= high:
        mid = low + (high - low)//2
        #print(f"mid is {mid} now")
        #print(f"value:{array[mid]-x}")
        if abs((array[mid])-x)< boundary:
            return mid
        elif ((array[mid])-x)< -(boundary):
            #print("mid<x")
            low = mid + 1
            #print(f"now lower bound is {low} and it's {array[low]}")
        else:
            #print("mid>x")
            high = mid - 1
            #print(f"now higher bound is {high} and it's {array[high]}")
    return -1

array = sortcomp[1]
callcomp = sortcomp[0]

x = 1714
boundary = 2
result = binarycompositionsearch(array, x, 0, len(array)-1, boundary)


#Waiting: Extend search to find out all 
#the index has a mass difference less than defined BOUNDARY

if result != -1:
    print("Element is present at index " + str(result))
    #go upper
    upsearch = result+1
    lowsearch = result-1
    if abs((array[upsearch])-x)< boundary:
        print("Element is ALSO present upper at index " + str(upsearch) + ":" + str(sortcomp[1][upsearch]))
        upsearch = result+1
    else:
        print("Stop expanding searching upper element")
    if abs((array[lowsearch])-x)< boundary:
        print("Element is present lower at index " + str(lowsearch)+ ":"  + str(sortcomp[1][lowsearch]))
        lowsearch = result-1
    else:
        print("Stop expanding searching lower element")
else:
    print("Not found")

Element is present at index 19
Stop expanding searching upper element
Element is present lower at index 18:1714.875


In [4]:
print(f"{array[15]}+{callcomp[15]}")

1684.864+(1, 3, 2, 1, 0, 0)


In [5]:
print(f"{array[16]}+{callcomp[16]}")

1688.871+(0, 4, 2, 0, 0, 1)


In [6]:
#this blcok is designed for looping through the spectra
def compileresult(result,boundary):
    foundindex = []
    interpretedmz = None
    interpretedcomp = None
    upper = True
    lower = True
    #print(f"result is {result}")
    if result != -1:
        interpretedmz = array[result]
        interpretedcomp = callcomp[result]
        foundindex.append((interpretedmz, interpretedcomp))
        interpretedmz = None
        interpretedcomp = None
        upsearch = result+1
        escapelower = False
        if result > 0:
            lowsearch = result-1
        else:
            lowsearch = 0
            escapelower = True
        print(foundindex)
        while abs((array[upsearch])-array[result])< boundary:
            #print("upper test")
            #print(f"upsearch{upsearch}")
            interpretedmz = array[upsearch]
            interpretedcomp = callcomp[upsearch]
            foundindex.append((interpretedmz, interpretedcomp))
            #print(f"debug mass diff{(array[upsearch]-array[result])} and index {upsearch} vs {result}")
            upsearch = upsearch+1
            #print(f"now the index is {upsearch}")
        #print("Stop expanding searching upper element")
        if escapelower is True:
            print("Trying to search a mass lower than in silico minimum")
        while abs((array[lowsearch])-array[result])< boundary and not escapelower:
            #print("lower test")
            #print(f"lowsearch{lowsearch}")
            interpretedmz = array[lowsearch]
            interpretedcomp = callcomp[lowsearch]
            foundindex.append((interpretedmz, interpretedcomp))
            lowsearch = lowsearch-1
            #print(f"debug mass diff{(array[upsearch]-array[result])}")
            #print(f"now the index is {lowsearch}")
        #print("Stop expanding searching lower element")
        #print(f"Found index: {foundindex}")
        return foundindex
    else:
        #print("None")
        return foundindex

result = binarycompositionsearch(array, x, 0, len(array)-1, boundary)
compileresult(result,boundary)

[(1714.875, '(1, 3, 2, 0, 1, 0)')]


[(1714.875, '(1, 3, 2, 0, 1, 0)'), (1714.875, '(0, 4, 2, 1, 0, 0)')]

In [8]:
#read the file
searchinput = pd.read_csv('zf_sPerMeNG_intestine MS2 summary from 48127 at 20230611-182546.csv', sep = '\t')
searchinputindex = searchinput["in [H+]"]
#set boundary
boundary = 2
foundindextmp = []
for i in range(100):
    print(f"mz from spectra file {searchinputindex[i]}")
    tmp = int(searchinputindex[i])
    result = binarycompositionsearch(array, tmp, 0, len(array)-1, boundary)
    compileresult(result,boundary)
    print("-"*10)

mz from spectra file 1596.5488951953123
----------
mz from spectra file 1234.6575652734375
----------
mz from spectra file 1720.456271328125
----------
mz from spectra file 1596.5494445117188
----------
mz from spectra file 1550.5018035546875
----------
mz from spectra file 1732.491427578125
[(1731.89, '(1, 5, 2, 0, 0, 0)')]
----------
mz from spectra file 1380.545138515625
----------
mz from spectra file 1744.5279266015625
[(1744.885, '(0, 4, 2, 0, 1, 0)')]
----------
mz from spectra file 1890.41452328125
[(1888.964, '(2, 3, 2, 0, 1, 0)')]
----------
mz from spectra file 1508.49850765625
----------
mz from spectra file 1180.6207000390625
----------
mz from spectra file 1392.579806484375
----------
mz from spectra file 1884.560430839844
[(1884.98, '(0, 3, 5, 0, 0, 0)')]
----------
mz from spectra file 1884.5607970507813
[(1884.98, '(0, 3, 5, 0, 0, 0)')]
----------
mz from spectra file 1884.5607970507813
[(1884.98, '(0, 3, 5, 0, 0, 0)')]
----------
mz from spectra file 1504.686373867187

In [9]:
#try to add the returned value into df
searchinput = pd.read_csv('zf_sPerMeNG_intestine MS2 summary from 48127 at 20230611-182546.csv', sep = '\t')
searchinputindex = searchinput["in [H+]"]
#set boundary
boundary = 2
foundindextmp = []
for i in range(len(searchinput)):
    print(f"mz from spectra file {searchinputindex[i]}")
    tmp = int(searchinputindex[i])
    result = binarycompositionsearch(array, tmp, 0, len(array)-1, boundary)
    foundindextmp.append(compileresult(result,boundary))
searchinput['Predicted composition'] = foundindextmp

searchinput.head(100)

mz from spectra file 1596.5488951953123
mz from spectra file 1234.6575652734375
mz from spectra file 1720.456271328125
mz from spectra file 1596.5494445117188
mz from spectra file 1550.5018035546875
mz from spectra file 1732.491427578125
[(1731.89, '(1, 5, 2, 0, 0, 0)')]
mz from spectra file 1380.545138515625
mz from spectra file 1744.5279266015625
[(1744.885, '(0, 4, 2, 0, 1, 0)')]
mz from spectra file 1890.41452328125
[(1888.964, '(2, 3, 2, 0, 1, 0)')]
mz from spectra file 1508.49850765625
mz from spectra file 1180.6207000390625
mz from spectra file 1392.579806484375
mz from spectra file 1884.560430839844
[(1884.98, '(0, 3, 5, 0, 0, 0)')]
mz from spectra file 1884.5607970507813
[(1884.98, '(0, 3, 5, 0, 0, 0)')]
mz from spectra file 1884.5607970507813
[(1884.98, '(0, 3, 5, 0, 0, 0)')]
mz from spectra file 1504.6863738671875
mz from spectra file 1424.63327328125
mz from spectra file 1436.66842953125
mz from spectra file 1007.4947845117188
mz from spectra file 1062.5379363671875
mz from

mz from spectra file 2185.1571990625
mz from spectra file 2301.24606625
[(2301.17, '(0, 7, 2, 0, 0, 1)')]
mz from spectra file 1525.8389617578125
mz from spectra file 1085.6351043359375
mz from spectra file 1547.92526546875
mz from spectra file 1042.6407195703125
mz from spectra file 2033.0560027734373
[(2033.042, '(3, 3, 2, 1, 0, 0)')]
mz from spectra file 1401.87936703125
mz from spectra file 1305.6719695703125
mz from spectra file 1105.636935390625
mz from spectra file 1846.994357265625
[(1845.945, '(0, 3, 2, 1, 0, 1)')]
mz from spectra file 1266.69577328125
mz from spectra file 2300.233126796875
[(2301.17, '(0, 7, 2, 0, 0, 1)')]
mz from spectra file 2188.10007015625
mz from spectra file 1402.808810390625
mz from spectra file 1069.5413543359375
mz from spectra file 2193.1298553125
[(2192.132, '(2, 4, 4, 0, 0, 0)')]
mz from spectra file 1355.7774383203125
[(1353.701, '(0, 4, 2, 0, 0, 0)')]
mz from spectra file 1978.017550625
[(1977.016, '(1, 5, 3, 0, 0, 0)')]
mz from spectra file 218

[(2501.263, '(2, 6, 2, 0, 1, 0)')]
mz from spectra file 2190.248263515625
mz from spectra file 4359.251165703125
[(4359.21, '(4, 4, 7, 3, 0, 0)')]
mz from spectra file 2791.445285
[(2789.407, '(2, 4, 2, 1, 1, 1)')]
mz from spectra file 2472.259005703125
[(2471.253, '(3, 5, 2, 0, 1, 0)')]
mz from spectra file 3137.6091979296875
[(3137.572, '(0, 4, 2, 2, 1, 2)')]
mz from spectra file 3460.761419609375
[(3458.79, '(2, 3, 10, 0, 0, 0)')]
mz from spectra file 2477.3601744921875
[(2478.285, '(2, 3, 6, 0, 0, 0)')]
mz from spectra file 1960.0296355859373
[(1960.001, '(0, 4, 3, 1, 0, 0)')]
mz from spectra file 1938.0400115625
mz from spectra file 2457.3154021875
[(2456.265, '(3, 4, 3, 0, 0, 1)')]
mz from spectra file 2154.1493865625
[(2155.111, '(0, 3, 2, 0, 0, 3)')]
mz from spectra file 2465.303439296875
[(2465.253, '(2, 3, 3, 2, 0, 0)')]
mz from spectra file 2640.368380703125
[(2641.322, '(1, 4, 2, 2, 1, 0)')]
mz from spectra file 2242.13327328125
[(2241.149, '(2, 5, 2, 0, 0, 1)')]
mz from sp

mz from spectra file 2460.310275234375
[(2458.244, '(0, 6, 2, 1, 0, 1)')]
mz from spectra file 2906.501193203125
[(2907.47, '(0, 7, 3, 1, 0, 1)')]
mz from spectra file 3250.7207701953125
[(3251.629, '(1, 4, 3, 1, 2, 1)')]
mz from spectra file 2368.232638515625
[(2366.221, '(3, 4, 4, 0, 0, 0)')]
mz from spectra file 2447.241915859375
[(2445.249, '(2, 6, 2, 0, 0, 1)')]
mz from spectra file 3488.76315609375
[(3487.731, '(0, 9, 2, 2, 1, 0)')]
mz from spectra file 2966.504122890625
[(2967.504, '(2, 9, 3, 0, 0, 0)')]
mz from spectra file 2070.172579921875
mz from spectra file 1582.7947723046875
mz from spectra file 2077.133517421875
[(2076.048, '(0, 4, 2, 2, 0, 0)')]
mz from spectra file 3153.617404140625
[(3154.588, '(1, 5, 2, 1, 1, 2)')]
mz from spectra file 1709.908908046875
mz from spectra file 2118.110079921875
[(2117.075, '(0, 3, 3, 2, 0, 0)')]
mz from spectra file 2976.553439296875
[(2976.492, '(0, 5, 2, 3, 0, 1)')]
mz from spectra file 2479.302706875
[(2478.285, '(2, 3, 6, 0, 0, 0)')

[(3498.747, '(0, 8, 3, 3, 0, 0)')]
mz from spectra file 3880.01215203125
[(3878.963, '(3, 4, 3, 1, 1, 3)')]
mz from spectra file 2184.14499203125
mz from spectra file 3488.780490078125
[(3487.731, '(0, 9, 2, 2, 1, 0)')]
mz from spectra file 2172.13815609375
[(2170.1, '(0, 8, 2, 0, 0, 0)')]
mz from spectra file 2844.459689296875
[(2843.442, '(3, 4, 3, 2, 0, 0)')]
mz from spectra file 3953.051458671875
[(3954.969, '(1, 3, 5, 2, 3, 0)')]
mz from spectra file 3677.919012382813
[(3676.879, '(1, 6, 5, 0, 0, 3)')]
mz from spectra file 3018.56882015625
[(3019.511, '(4, 3, 2, 0, 3, 0)')]
mz from spectra file 3704.9028991015625
[(3702.859, '(4, 5, 2, 2, 1, 1)')]
mz from spectra file 2658.47897640625
[(2658.324, '(0, 3, 2, 0, 3, 1)')]
mz from spectra file 1781.9407683984375
mz from spectra file 3470.75385125
[(3470.763, '(2, 3, 3, 2, 0, 3)')]
mz from spectra file 2931.5842345507813
[(2931.481, '(1, 3, 4, 0, 2, 1)')]
mz from spectra file 2288.208224453125
mz from spectra file 2152.111300625
[(2151

...

[(3765.906, '(2, 4, 6, 3, 0, 0)')]
mz from spectra file 3936.91012875
[(3934.977, '(3, 4, 3, 1, 2, 2)')]
mz from spectra file 5157.469061210937
[(5155.556, '(0, 9, 3, 3, 2, 2)')]
mz from spectra file 5195.46036671875
[(5196.595, '(4, 8, 4, 2, 2, 1)')]
mz from spectra file 3470.694552578125
[(3470.763, '(2, 3, 3, 2, 0, 3)')]
mz from spectra file 5184.532415703125
[(5184.633, '(5, 3, 9, 2, 1, 1)')]
mz from spectra file 2816.36178890625
[(2817.425, '(0, 3, 3, 1, 1, 2)')]
mz from spectra file 3870.90573421875
[(3868.958, '(5, 3, 5, 2, 1, 0)')]
mz from spectra file 3915.927462734375
[(3915.947, '(4, 4, 2, 3, 2, 0)')]
mz from spectra file 3612.744845546875
[(3610.811, '(1, 5, 5, 1, 2, 0)')]
mz from spectra file 3801.8427459375
[(3799.9, '(4, 4, 3, 2, 2, 0)')]
mz from spectra file 3892.850314296875
[(3893.987, '(3, 7, 5, 0, 0, 2)')]
mz from spectra file 1922.955783046875
mz from spectra file 5760.8231871875
[(5759.899, '(2, 6, 8, 0, 3, 3)')]
mz from spectra file 2122.1083709375
[(2123.074, '(

mz from spectra file 2059.031954921875
[(2059.069, '(1, 3, 5, 0, 0, 0)')]
mz from spectra file 3145.6079772265625
[(3146.597, '(0, 4, 5, 2, 0, 1)')]
mz from spectra file 2285.1181365625
[(2284.155, '(0, 5, 2, 0, 1, 1)')]
mz from spectra file 3554.827853359375
[(3554.797, '(2, 4, 5, 0, 2, 1)')]
mz from spectra file 3775.827365078125
[(3775.912, '(2, 7, 5, 0, 1, 1)')]
mz from spectra file 3542.75534359375
[(3543.781, '(1, 6, 4, 0, 2, 1)')]
mz from spectra file 2551.230685390625
[(2549.322, '(1, 3, 7, 0, 0, 0)')]
mz from spectra file 3503.746798671875
[(3502.753, '(0, 4, 2, 1, 2, 3)')]
mz from spectra file 2844.392306484375
[(2843.442, '(3, 4, 3, 2, 0, 0)')]
mz from spectra file 5184.533148125
[(5184.633, '(5, 3, 9, 2, 1, 1)')]
mz from spectra file 3474.71725765625
[(3474.771, '(2, 7, 4, 0, 0, 2)')]
mz from spectra file 3979.93356625
[(3978.019, '(1, 6, 6, 0, 1, 2)')]
mz from spectra file 2294.072970546875
[(2295.184, '(4, 4, 3, 0, 0, 0)')]
mz from spectra file 2246.1161834375
[(2246.154,

mz from spectra file 3864.863497890625
[(3863.928, '(2, 6, 3, 0, 3, 1)')]
mz from spectra file 5219.453924492187
[(5220.606, '(2, 7, 5, 4, 1, 1)')]
mz from spectra file 2886.415255703125
[(2886.448, '(1, 4, 3, 2, 1, 0)')]
mz from spectra file 5790.861273125
[(5789.91, '(0, 8, 8, 1, 2, 3)')]
mz from spectra file 3852.887423671875
[(3852.947, '(0, 7, 4, 0, 1, 3)')]
mz from spectra file 1063.6875945703125
mz from spectra file 3837.857638515625
[(3836.943, '(2, 3, 7, 2, 1, 0)')]
mz from spectra file 3634.771701015625
[(3633.836, '(4, 3, 2, 0, 2, 3)')]
mz from spectra file 2756.387823417969
[(2755.413, '(1, 3, 5, 1, 0, 1)')]
mz from spectra file 4233.078680351562
[(4234.125, '(1, 3, 5, 2, 2, 2)')]
mz from spectra file 3661.79049984375
[(3661.856, '(2, 6, 4, 0, 1, 2)')]
mz from spectra file 5388.604559257812
[(5386.705, '(5, 3, 8, 2, 3, 0)')]
mz from spectra file 3984.88913265625
[(3983.037, '(2, 5, 9, 1, 0, 0)')]
mz from spectra file 3761.859103359375
[(3762.867, '(1, 4, 2, 0, 4, 2)')]
mz f

mz from spectra file 6616.271401875
[(6615.309, '(2, 8, 7, 3, 2, 3)')]
mz from spectra file 3560.7754576953125
[(3561.806, '(2, 3, 6, 3, 0, 0)')]
mz from spectra file 3929.951144375
[(3930.958, '(0, 7, 4, 3, 1, 0)')]
mz from spectra file 3796.84762875
[(3796.898, '(1, 8, 2, 0, 2, 2)')]
mz from spectra file 4179.029974296875
[(4180.128, '(0, 9, 8, 0, 0, 1)')]
mz from spectra file 3822.82419125
[(3820.921, '(2, 4, 2, 2, 1, 3)')]
mz from spectra file 1078.818331875
mz from spectra file 4194.029974296875
[(4193.112, '(3, 6, 4, 3, 0, 1)')]
mz from spectra file 4410.103216484375
[(4410.219, '(4, 4, 3, 4, 0, 2)')]
mz from spectra file 3442.6708709375
[(3442.733, '(3, 5, 3, 0, 2, 1)')]
mz from spectra file 3542.768771328125
[(3543.781, '(1, 6, 4, 0, 2, 1)')]
mz from spectra file 3428.6562225
[(3427.697, '(1, 4, 2, 0, 4, 1)')]
mz from spectra file 3575.73092953125
[(3575.784, '(1, 9, 3, 0, 2, 0)')]
mz from spectra file 3936.919161953125
[(3934.977, '(3, 4, 3, 1, 2, 2)')]
mz from spectra file 28

,entry no,MS1scan no,MS1Isolation mass,MS1monoIsomass,chargeState,in [H+],intensityN\/A,StructureN\/A,MS2 Scan no,peaklist,Predicted composition
0,1,164,532.854858,532.854858,3,1596.548895,ext from peak list,structure na,166,"MS2peaklist(dMass=(94.03176879882812, 95.39357...",[]
1,2,180,617.832703,617.832703,2,1234.657565,ext from peak list,structure na,182,"MS2peaklist(dMass=(90.79227447509766, 92.70963...",[]
2,3,183,860.732056,860.732056,2,1720.456271,ext from peak list,structure na,185,"MS2peaklist(dMass=(91.41570281982422, 92.77639...",[]
3,4,188,532.855042,532.855042,3,1596.549445,ext from peak list,structure na,190,"MS2peaklist(dMass=(90.06702423095703, 92.66555...",[]
4,5,191,775.754822,775.754822,2,1550.501804,ext from peak list,structure na,193,"MS2peaklist(dMass=(92.60694122314453, 92.63258...",[]
...,...,...,...,...,...,...,...,...,...,...,...
95,96,1969,663.908264,663.406067,2,1325.804294,ext from peak list,structure na,1996,"MS2peaklist(dMass=(93.54866790771484, 96.24431...","[(1323.69, (1, 3, 2, 0, 0, 0))]"
96,97,1969,747.440796,747.440796,2,1493.873752,ext from peak list,structure na,1997,"MS2peaklist(dMass=(94.32759857177734, 112.2564...",[]
97,98,1969,733.916565,733.916565,2,1466.825290,ext from peak list,structure na,1998,"MS2peaklist(dMass=(101.18326568603516, 101.728...",[]
98,99,1969,885.469604,885.469604,2,1769.931369,ext from peak list,structure na,1999,"MS2peaklist(dMass=(92.07662963867188, 94.25292...",[]


In [10]:
#save the file into another file and generate a setting file... does setting file needed?
timestamp = time.strftime("%Y%m%d-%H%M%S") #datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
extfilename= "zf_sPerMeNG_intestine MS2 summary from 48127 at 20230611-182546 " + timestamp + " predictedcomp.csv"
searchinput.to_csv(extfilename, sep = '\t')
print("export finished") #Doesn't check the existence of file, should be modified in the future

export finished


In [9]:
#find the folder contains *predictcomp.csv
import fnmatch
import os

for file in os.listdir('.'):
    if fnmatch.fnmatch(file, '*predictedcomp.csv'):
        print(file)
#https://docs.python.org/3/library/fnmatch.html#module-fnmatch

zf_sPerMeNG_brain MS2 summary from 44218 at 20230307-153550 20230507-181703 predictedcomp.csv
zf_sPerMeNG_brain MS2 summary from 44218 at 20230307-153550 20230511-143124 predictedcomp.csv
